In [4]:
import pandas as pd
from pathlib import Path
import re

INPUT_CSV = "../results/csv_triptyques/manuel_chap1to5.csv"
INPUT_TXT = "../data/PROPP/new_txt/tdm_auto_chap1to5.txt"
OUTPUT_CSV_ENRICHI = "../results/csv_triptyques/manuel_chap1to5_chap.csv"


In [5]:
df = pd.read_csv(INPUT_CSV)

In [6]:
# ---------- Reconstruction des chapitres depuis le .txt ----------
txt = Path(INPUT_TXT).read_text(encoding="utf-8")

# Les chapitres du fichier texte sont séparés par de grands blancs (>= 4 sauts de ligne)
chapter_blocks = re.split(r'(?:\n\s*){4,}', txt.strip())

def normalize_for_match(s):
    s = str(s).lower()
    s = s.replace("\xa0", " ").replace("’", "'").replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s).strip()
    # normalisation agressive pour éviter les écarts de tokenisation du CSV
    s = re.sub(r"[^a-zà-ÿ0-9]+", "", s)
    return s

chapter_norm = [normalize_for_match(block) for block in chapter_blocks]

# table des phrases uniques
sent_df = (
    df[["Sentence_index", "Phrase"]]
    .drop_duplicates()
    .sort_values("Sentence_index")
    .reset_index(drop=True)
)

sentence_to_chapter = {}
current_chapter = 0

for _, row in sent_df.iterrows():
    sent_idx = int(row["Sentence_index"])
    phrase_norm = normalize_for_match(row["Phrase"])

    found = False
    for chap_i in range(current_chapter, len(chapter_norm)):
        if phrase_norm and phrase_norm in chapter_norm[chap_i]:
            sentence_to_chapter[sent_idx] = chap_i + 1
            current_chapter = chap_i
            found = True
            break

    if not found:
        sentence_to_chapter[sent_idx] = pd.NA

df["chapter"] = df["Sentence_index"].map(sentence_to_chapter)

# sauvegarde du CSV enrichi dès le début
df.to_csv(OUTPUT_CSV_ENRICHI, index=False, encoding="utf-8")

print("Nombre de chapitres détectés dans le .txt :", len(chapter_blocks))
print("Phrases sans chapitre attribué :", int(df[['Sentence_index','chapter']].drop_duplicates()['chapter'].isna().sum()))
print("CSV enrichi sauvegardé :", Path(OUTPUT_CSV_ENRICHI).resolve())

df.head()

Nombre de chapitres détectés dans le .txt : 5
Phrases sans chapitre attribué : 0
CSV enrichi sauvegardé : C:\Users\arnod\stage_lattice\Model_TDM\results\csv_triptyques\manuel_chap1to5_chap.csv


,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,personnages_objet,personnages_triptyque,lieux_sujet,lieux_objet,lieux_triptyque,types_lieux_triptyque,a_personnage_triptyque,a_lieu_triptyque,a_personnage_et_lieu_triptyque,chapter
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,NaN,acl,obj,NaN,8,10.0,...,"['5', '10']","['5', '10']",[],['5'],['5'],['FAC'],True,True,True,1
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,nsubj,acl:relcl,obl:mod,23.0,24,26.0,...,[],['77'],[],[],[],[],True,False,False,1
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l'année 1872,nsubj:pass,root,obl:mod,7.0,30,3.0,...,[],['5'],[],[],[],[],True,False,False,1
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,nsubj:pass,root,obl:agent,7.0,30,32.0,...,['0'],"['5', '0']",[],[],[],[],True,False,False,1
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l'un des membres les plus singuliers et les pl...,nsubj:pass,root,obl:mod,7.0,30,39.0,...,"['0', '3', '9']","['5', '0', '3', '9']",[],[],[],[],True,False,False,1
